In [1]:
# Import von Bibliotheken
import urllib.request, urllib.parse, urllib.error
import io,re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup

**1. Teil: Wahlprogramme**

In [2]:
# Einlesen der Langwahlprogramme (aus JSON-Format in ein Wörterbuch)

# für CDU/CSU
with open("cdu-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    CDU_text_json = json.load(datei)

# für SPD
with open("spd-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    SPD_text_json = json.load(datei)

# für Die Linke
with open("linke-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    Linke_text_json = json.load(datei)

# für die AfD
with open("afd-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    AfD_text_json = json.load(datei)

# für Bündnis 90/Grüne
with open("gruene-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    Grüne_text_json = json.load(datei)

# für FDP
with open("fdp-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    FDP_text_json = json.load(datei)

In [3]:
# Überführen aller 'eigentlichen' Texte in eine Liste pro Partei, d.h. ohne Kapitelüberschriften

# Suchfunktion in den "Partei-Wörterbüchern"

def finde_gefilterte_texte(struktur):
    ergebnisse = []
    
    # Fall A: Wenn das aktuelle Element ein Dictionary ist
    if isinstance(struktur, dict):
        # Typ/Art auslesen (liefert None, wenn der Schlüssel fehlt)
        aktueller_typ = struktur.get('type') or struktur.get('art')            # nur Suchen in diesen Elementen
        
        # Prüfen, ob 'text' existiert und der gefundene Typ gültig ist
        if 'text' in struktur and aktueller_typ in ['paragraph', 'bullet', 'absatz']:   # nur Texte überführen mit diesem Typ
            ergebnisse.append(struktur['text'])
        
        # Tiefer in alle Werte schauen für eventuelle Verschachtelungen
        for wert in struktur.values():
            ergebnisse.extend(finde_gefilterte_texte(wert))
                
    # Fall B: Wenn das aktuelle Element eine Liste ist
    elif isinstance(struktur, list):
        for element in struktur:
            ergebnisse.extend(finde_gefilterte_texte(element))
            
    return ergebnisse

CDU_text = finde_gefilterte_texte(CDU_text_json)
SPD_text = finde_gefilterte_texte(SPD_text_json)
Linke_text = finde_gefilterte_texte(Linke_text_json)
Grüne_text = finde_gefilterte_texte(Grüne_text_json)
AfD_text = finde_gefilterte_texte(AfD_text_json)
FDP_text = finde_gefilterte_texte(FDP_text_json)

In [4]:
# Überführen aller 'eigentlichen' Texte in einen String pro Partei, d.h. ohne Kapitelüberschriften

# für CDU/CSU
CDU_text_str = [text if text.endswith('.') else text + '.' for text in CDU_text]   # ein "Punkt" wird gesetzt bei Bulletpoints
CDU_text = " ".join(CDU_text_str)

# für SPD
SPD_text_str = [text if text.endswith('.') else text + '.' for text in SPD_text]
SPD_text = " ".join(SPD_text_str)

# für Die Linke
Linke_text_str = [text if text.endswith('.') else text + '.' for text in Linke_text]
Linke_text = " ".join(Linke_text_str)

# für die AfD
AfD_text_str = [text if text.endswith('.') else text + '.' for text in AfD_text]
AfD_text = " ".join(AfD_text_str)

# für Bündnis 90/Grüne
Grüne_text_str = [text if text.endswith('.') else text + '.' for text in Grüne_text]
Grüne_text = " ".join(Grüne_text_str)

# für FDP
FDP_text_str = [text if text.endswith('.') else text + '.' for text in FDP_text]
FDP_text = " ".join(FDP_text_str)

In [5]:
# Bereinigung der Texte - TEIL 1 -> um Sonderzeichen usw., siehe Kommentar neben entsprechendem reg-Ausdruck

def str_bereinigen(i):
    i = re.sub(r'-\s+', '', i)                               # Ersetzt den Bindestrich gefolgt von einem oder mehreren Leerzeichen durch nichts
    i = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', i)            # Sonderzeichen-Filter, löscht alles, was nicht ausdrücklich erhalten bleiben soll
    i = re.sub(r'([A-Za-zÄÖÜäöüß])\1{3,}', r'\1\1\1', i)     # Begrenzung von extremen Wiederholungen
    i = re.sub(r'\s+', ' ', i).strip()                       # Normalisierung von Whitespaces (ersetzt durch ein Leerzeichen)
    return i
    
CDU_text = str_bereinigen(CDU_text)
SPD_text = str_bereinigen(SPD_text)
Grüne_text = str_bereinigen(Grüne_text)
AfD_text = str_bereinigen(AfD_text)
Linke_text = str_bereinigen(Linke_text)
FDP_text = str_bereinigen(FDP_text)

In [6]:
# Bereinigen der zuvor bereinigten Texte - TEIL 2 => Umformen von Abkürzungen in Text um Eindruck von einem Satzende zu vermeiden

def bereinige_abkuerzungen(text):
    # Dictionary mit den Top 20 Ersetzungen (Regex-Muster als Key)
    ersetzungen = {
        r'\bz\.\s*B\.': 'zum Beispiel',   # r'\bz\.\s*B\.\b': 'zum Beispiel',
        r'\bu\.\s*a\.': 'unter anderem',
        r'\bd\.\s*h\.': 'das heißt',
        r'\bbzw\.': 'beziehungsweise',
        r'\bbzw': 'beziehungsweise',   # Variante ohne Punkt
        r'\bsog\.': 'sogenannte',
        r'\bca\.': 'circa',
        r'\bevtl\.': 'eventuell',
        r'\binkl\.': 'inklusive',
        r'\betc\.': 'et cetera',
        r'\bvgl\.': 'vergleiche',
        r'\bs\.': 'siehe',
        r'\bu\.\s*v\.\s*m\.': 'und vieles mehr',
        r'\bu\.\s*ä\.': 'und ähnliche',
        r'\bggf\.': 'gegebenenfalls',
        r'\bzzgl\.': 'zuzüglich',
        r'\bebd\.': 'ebenda',
        r'\bo\.\s*g\.': 'oben genannte',
        r'\bu\.\s*g\.': 'unten genannte',
        r'\bi\.\s*d\.\s*R\.': 'in der Regel',
        r'\bv\.\s*a\.': 'vor allem',

        # 2. Titel & Personen (Verhindern Satzabbruch mitten im Fluss)
        r'\bDr\.': 'Doktor',
        r'\bProf\.': 'Professor',
        r'\bFr\.': 'Frau',
        r'\bHr\.': 'Herr',

        # 3. Wortlaengen- & Silben-Verfaelscher
        r'\bbspw\.': 'beispielsweise',
        r'\bbspw': 'beispielsweise',   # Variante ohne Punkt
        r'\bbsp\.': 'Beispiel',
        r'\bJh\.': 'Jahrhundert',
        r'\bMio\.': 'Millionen',
        r'\bMrd\.': 'Milliarden',
        r'\bbetr\.': 'betreffend',
        r'\bbezgl\.': 'bezüglich',
        r'\bvs\.': 'versus'
    }

    # Text Schritt für Schritt bereinigen
    for muster, ersetzung in ersetzungen.items():
        # flags=re.IGNORECASE sorgt dafür, dass auch "Z.B." oder "Bzw." gefunden werden
        text = re.sub(muster, ersetzung, text, flags=re.IGNORECASE)

    return text

CDU_text = bereinige_abkuerzungen(CDU_text)
SPD_text = bereinige_abkuerzungen(SPD_text)
Grüne_text = bereinige_abkuerzungen(Grüne_text)
AfD_text = bereinige_abkuerzungen(AfD_text)
Linke_text = bereinige_abkuerzungen(Linke_text)
FDP_text = bereinige_abkuerzungen(FDP_text)

In [7]:
# Extrahiert alle Wörter und separaten Zahlen (ohne Punkt, Komma etc.) aus den zuvor vorverarbeiteten Texten
# zwecks Ermittlung der Anzahl der Wörter pro Parteiprogramm
# Annahme: separate Zahlen werden bei der statistischen Wortzählung in Texten wie hier als eigene Wörter gezählt

CDU_WP_anzahl = re.findall(r'[a-zA-ZäöüÄÖÜß0-9]+(?:-[a-zA-ZäöüÄÖÜß0-9]+)*', CDU_text)
SPD_WP_anzahl = re.findall(r'[a-zA-ZäöüÄÖÜß0-9]+(?:-[a-zA-ZäöüÄÖÜß0-9]+)*', SPD_text)
AfD_WP_anzahl = re.findall(r'[a-zA-ZäöüÄÖÜß0-9]+(?:-[a-zA-ZäöüÄÖÜß0-9]+)*', AfD_text)
Linke_WP_anzahl = re.findall(r'[a-zA-ZäöüÄÖÜß0-9]+(?:-[a-zA-ZäöüÄÖÜß0-9]+)*', Linke_text)
Grüne_WP_anzahl = re.findall(r'[a-zA-ZäöüÄÖÜß0-9]+(?:-[a-zA-ZäöüÄÖÜß0-9]+)*', Grüne_text)
FDP_WP_anzahl = re.findall(r'[a-zA-ZäöüÄÖÜß0-9]+(?:-[a-zA-ZäöüÄÖÜß0-9]+)*', FDP_text)

In [8]:
# Berechnung der Anzahl der Wörter pro Parteiprogramm

CDU_WP_anzahl = len(CDU_WP_anzahl)
SPD_WP_anzahl = len(SPD_WP_anzahl)
AfD_WP_anzahl = len(AfD_WP_anzahl)
Linke_WP_anzahl = len(Linke_WP_anzahl)
Grüne_WP_anzahl = len(Grüne_WP_anzahl)
FDP_WP_anzahl = len(FDP_WP_anzahl)

**2. Teil: Bundestagsreden**

In [9]:
# Aufruf Stammdatendatei, die alle Mitglieder des Bundestages seit 1949 aufführt / aufführen soll
# zwecks Extraktion der Parteizugehörigkeit der Redner

stammdaten = ("MDB_STAMMDATEN.XML")

with open(stammdaten, "r", encoding="utf-8") as f:
  	soup2 = BeautifulSoup(f, "xml")

redner_id_mdb = dict()
for mdb in soup2.find_all("MDB"):
	id_tag = mdb.find("ID")
	party_tag = mdb.find("PARTEI_KURZ")
	mdb_id = id_tag.text if id_tag else None
	party = party_tag.text if party_tag else None
	redner_id_mdb[mdb_id] = party

# da wo keine Parteizugehörigkeit aus den Stammdaten ersichtlich war, manuelles Hinzufügen diverser Nummern
redner_id_mdb["11002735"]  # MERZ
redner_id_mdb["999990151"] = "SPD"
redner_id_mdb["999990133"] = "SPD"
redner_id_mdb["999990074"] = "SPD"
redner_id_mdb["11005217 999990074"] = "SPD"
redner_id_mdb["999990119"] = "SPD"
redner_id_mdb["999990149"] = "SPD"
redner_id_mdb["999990080"] = "parteilos"
redner_id_mdb["999990142"] = "parteilos"
redner_id_mdb["999990154"] = "parteilos"
redner_id_mdb["999990152"] = "CDU"
redner_id_mdb["999990153"] = "CDU"
redner_id_mdb["999990150"] = "CDU"
redner_id_mdb["999990141"] = "CSU"
redner_id_mdb["999990193"] = "SPD"
redner_id_mdb["999990093"] = "SPD"
redner_id_mdb["999990120"] = "SPD"
redner_id_mdb["999990129"] = "SPD"
redner_id_mdb["999990145"] = "SPD"
redner_id_mdb["999990144"] = "CDU"
redner_id_mdb["999990125"] = "CDU"
redner_id_mdb["999990147"] = "CSU"
redner_id_mdb["999990146"] = "SPD"
redner_id_mdb["999990121"] = "SPD"
redner_id_mdb["999990148"] = "SPD"
redner_id_mdb["999990122"] = "BÜNDNIS 90/DIE GRÜNEN"
redner_id_mdb["999990123"] = "DIE LINKE"
redner_id_mdb["999990148"] = "SPD"
redner_id_mdb["999990122"] = "BÜNDNIS 90/DIE GRÜNEN"
redner_id_mdb["999990123"] = "DIE LINKE"
redner_id_mdb["999990124"] = "SPD"
redner_id_mdb["999990078"] = "FDP"
redner_id_mdb["999990082"] = "SPD"

In [10]:
# Import der Bundestagsreden via gültigem API-Key für die 5 Betrachtungszeiträume
# API-Key gültig bis zunächst Ende Mai 2027
api = "R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ"

# Eingrenzung Zeitraum Nr. 1, Umwandlung in json-Format & Extraktion der XML-URLs (=Protokolle) aus dem Dictionary
html_zeitraum_1 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2022-01-01&f.datum.end=2022-03-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_1 = json.loads(html_zeitraum_1)
# Erfassen der XML-URLs in einer Liste
data_zeitraum_1_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_1["documents"]
    if "xml_url" in doc["fundstelle"]
]   # 17 Protokolle

# Eingrenzung Zeitraum Nr. 2...
html_zeitraum_2 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2023-10-01&f.datum.end=2023-12-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_2 = json.loads(html_zeitraum_2)
data_zeitraum_2_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_2["documents"]
    if "xml_url" in doc["fundstelle"]
]  # 19 Protokolle

# Eingrenzung Zeitraum Nr. 3...
html_zeitraum_3 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2024-10-01&f.datum.end=2024-12-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_3 = json.loads(html_zeitraum_3)
data_zeitraum_3_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_3["documents"]
    if "xml_url" in doc["fundstelle"]
]   # 19 Protokolle

# Eingrenzung Zeitraum Nr. 4...
html_zeitraum_4 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2025-01-01&f.datum.end=2025-02-23&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_4 = json.loads(html_zeitraum_4)
data_zeitraum_4_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_4["documents"]
    if "xml_url" in doc["fundstelle"]
]    # 4 Protokolle

# Eingrenzung Zeitraum Nr. 5...
html_zeitraum_5 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2025-05-01&f.datum.end=2025-07-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_5 = json.loads(html_zeitraum_5)
data_zeitraum_5_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_5["documents"]
    if "xml_url" in doc["fundstelle"]   
]    # 18 Protokolle

In [11]:
# Anzahl Reden pro Partei und Zeitraum
# zunächst leere Listen
text_liste_CDU_1 = []
text_liste_CDU_2 = []
text_liste_CDU_3 = []
text_liste_CDU_4 = []
text_liste_CDU_5 = []

text_liste_SPD_1 = []
text_liste_SPD_2 = []
text_liste_SPD_3 = []
text_liste_SPD_4 = []
text_liste_SPD_5 = []

text_liste_FDP_1 = []
text_liste_FDP_2 = []
text_liste_FDP_3 = []
text_liste_FDP_4 = []
text_liste_FDP_5 = []

text_liste_Grüne_1 = []
text_liste_Grüne_2 = []
text_liste_Grüne_3 = []
text_liste_Grüne_4 = []
text_liste_Grüne_5 = []

text_liste_Linke_1 = []
text_liste_Linke_2 = []
text_liste_Linke_3 = []
text_liste_Linke_4 = []
text_liste_Linke_5 = []

text_liste_AfD_1 = []
text_liste_AfD_2 = []
text_liste_AfD_3 = []
text_liste_AfD_4 = []
text_liste_AfD_5 = []

# Durchlauf aller extrahierter XML-URLs pro Zeitraum (1 bis 5), Anhängen der Reden pro Partei und Zeitraum an die obigen Listen
for i in data_zeitraum_1_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_1.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_1.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_1.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_1.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_1.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_1.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_1.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_1.append(p)

for i in data_zeitraum_2_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_2.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_2.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_2.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_2.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_2.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_2.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_2.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_2.append(p)

for i in data_zeitraum_3_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_3.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_3.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_3.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_3.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_3.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_3.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_3.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_3.append(p)

for i in data_zeitraum_4_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_4.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_4.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_4.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_4.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_4.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_4.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_4.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_4.append(p)

for i in data_zeitraum_5_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_5.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_5.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_5.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_5.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_5.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_5.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_5.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_5.append(p)

# Anzahl Reden pro Partei für Zeitraum 1
text_liste_CDU_1_no = len(text_liste_CDU_1)
text_liste_SPD_1_no = len(text_liste_SPD_1)
text_liste_AfD_1_no = len(text_liste_AfD_1)
text_liste_Linke_1_no = len(text_liste_Linke_1)
text_liste_Grüne_1_no = len(text_liste_Grüne_1)
text_liste_FDP_1_no = len(text_liste_FDP_1)

# Anzahl Reden pro Partei für Zeitraum 2
text_liste_CDU_2_no = len(text_liste_CDU_2)
text_liste_SPD_2_no = len(text_liste_SPD_2)
text_liste_AfD_2_no = len(text_liste_AfD_2)
text_liste_Linke_2_no = len(text_liste_Linke_2)
text_liste_Grüne_2_no = len(text_liste_Grüne_2)
text_liste_FDP_2_no = len(text_liste_FDP_2)

# Anzahl Reden pro Partei für Zeitraum 3
text_liste_CDU_3_no = len(text_liste_CDU_3)
text_liste_SPD_3_no = len(text_liste_SPD_3)
text_liste_AfD_3_no = len(text_liste_AfD_3)
text_liste_Linke_3_no = len(text_liste_Linke_3)
text_liste_Grüne_3_no = len(text_liste_Grüne_3)
text_liste_FDP_3_no = len(text_liste_FDP_3)

# Anzahl Reden pro Partei für Zeitraum 4
text_liste_CDU_4_no = len(text_liste_CDU_4)
text_liste_SPD_4_no = len(text_liste_SPD_4)
text_liste_AfD_4_no = len(text_liste_AfD_4)
text_liste_Linke_4_no = len(text_liste_Linke_4)
text_liste_Grüne_4_no = len(text_liste_Grüne_4)
text_liste_FDP_4_no = len(text_liste_FDP_4)

# Anzahl Reden pro Partei für Zeitraum 5
text_liste_CDU_5_no = len(text_liste_CDU_5)
text_liste_SPD_5_no = len(text_liste_SPD_5)
text_liste_AfD_5_no = len(text_liste_AfD_5)
text_liste_Linke_5_no = len(text_liste_Linke_5)
text_liste_Grüne_5_no = len(text_liste_Grüne_5)
text_liste_FDP_5_no = len(text_liste_FDP_5)

In [12]:
# Anzahl der Wörter in 4. Periode zum Vergleich -> zwischen dem 4. und 5. Betrachtungszeitraum erschienen auch die Wahlprogramme
# zunächst leere Listen
text_liste_CDU_4_clean = []
text_liste_SPD_4_clean = []
text_liste_AfD_4_clean = []
text_liste_Linke_4_clean = []
text_liste_Grüne_4_clean = []
text_liste_FDP_4_clean = []

# Durchlaufen der "Partei"-Listen für Zeitraum Nr. 4, die die Anzahl Reden für diese Periode enthalten
# und Extraktion der "reinen Redetexte" + deren anschließende Bereinigung
import re
for i in text_liste_CDU_4:
  i = str(i[1:]).replace('<p klasse="J_1">', "")  # Entfernen von '<p klasse="J_1">' aus dem Redetext
  i = i.replace('<p klasse="J">', "")
  i = i.replace('<p klasse="O">', "")
  i = i.replace("<p klasse=", "")
  i = re.sub(r'-\s+', '', i)   
  i = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', i) 
  i = re.sub(r"\s+", " ", i)
  i = i.replace("\xa0", "")
  i = i.replace("</p>", "")
  i = i.replace("</p", "")
  text_liste_CDU_4_clean.append(i)

for i in text_liste_SPD_4:
  i = str(i[1:]).replace('<p klasse="J_1">', "")
  i = i.replace('<p klasse="J">', "")
  i = i.replace('<p klasse="O">', "")
  i = i.replace("<p klasse=", "")
  i = re.sub(r"\s+", " ", i)
  i = re.sub(r'-\s+', '', i)   
  i = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', i)  
  i = i.replace("\xa0", "")
  i = i.replace("</p>", "")
  i = i.replace("</p", "")
  text_liste_SPD_4_clean.append(i)

for i in text_liste_Linke_4:
  i = str(i[1:]).replace('<p klasse="J_1">', "")
  i = i.replace('<p klasse="J">', "")
  i = i.replace('<p klasse="O">', "")
  i = i.replace("<p klasse=", "")
  i = re.sub(r"\s+", " ", i)
  i = re.sub(r'-\s+', '', i)    
  i = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', i)  
  i = i.replace("\xa0", "")
  i = i.replace("</p>", "")
  i = i.replace("</p", "")
  text_liste_Linke_4_clean.append(i)

for i in text_liste_AfD_4:
  i = str(i[1:]).replace('<p klasse="J_1">', "")
  i = i.replace('<p klasse="J">', "")
  i = i.replace('<p klasse="O">', "")
  i = i.replace("<p klasse=", "")
  i = re.sub(r'-\s+', '', i)   
  i = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', i) 
  i = re.sub(r"\s+", " ", i)
  i = i.replace("\xa0", "")
  i = i.replace("</p>", "")
  i = i.replace("</p", "")
  text_liste_AfD_4_clean.append(i)


for i in text_liste_Grüne_4:
  i = str(i[1:]).replace('<p klasse="J_1">', "")
  i = i.replace('<p klasse="J">', "")
  i = i.replace('<p klasse="O">', "")
  i = i.replace("<p klasse=", "")
  i = re.sub(r"\s+", " ", i)
  i = re.sub(r'-\s+', '', i)   
  i = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', i)
  i = i.replace("\xa0", "")
  i = i.replace("</p>", "")
  i = i.replace("</p", "")
  text_liste_Grüne_4_clean.append(i)

for i in text_liste_FDP_4:
  i = str(i[1:]).replace('<p klasse="J_1">', "")
  i = i.replace('<p klasse="J">', "")
  i = i.replace('<p klasse="O">', "")
  i = i.replace("<p klasse=", "")
  i = re.sub(r'-\s+', '', i)   
  i = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', i) 
  i = re.sub(r"\s+", " ", i)
  i = i.replace("\xa0", "")
  i = i.replace("</p>", "")
  i = i.replace("</p", "")
  text_liste_FDP_4_clean.append(i)

In [13]:
# aus jeder Liste einen großen String pro Partei und Betrachtungszeitraum bilden

text_liste_CDU_4_str = "".join(text_liste_CDU_4_clean)
text_liste_SPD_4_str = "".join(text_liste_SPD_4_clean)
text_liste_AfD_4_str = "".join(text_liste_AfD_4_clean)
text_liste_Linke_4_str = "".join(text_liste_Linke_4_clean)
text_liste_Grüne_4_str = "".join(text_liste_Grüne_4_clean)
text_liste_FDP_4_str = "".join(text_liste_FDP_4_clean)

# Extrahiert alle Wörter und separaten Zahlen (ohne Punkt, Komma etc.) -> Anzahl
# s.o. Annahme: separate Zahlen werden bei der statistischen Wortzählung in Texten wie hier als eigene Wörter gezählt

text_liste_CDU_4_str_bereinigt = re.findall(r'[a-zA-ZäöüÄÖÜß0-9]+(?:-[a-zA-ZäöüÄÖÜß0-9]+)*', text_liste_CDU_4_str)
text_liste_SPD_4_str_bereinigt = re.findall(r'[a-zA-ZäöüÄÖÜß0-9]+(?:-[a-zA-ZäöüÄÖÜß0-9]+)*', text_liste_SPD_4_str)
text_liste_AfD_4_str_bereinigt = re.findall(r'[a-zA-ZäöüÄÖÜß0-9]+(?:-[a-zA-ZäöüÄÖÜß0-9]+)*', text_liste_AfD_4_str)
text_liste_Linke_4_str_bereinigt = re.findall(r'[a-zA-ZäöüÄÖÜß0-9]+(?:-[a-zA-ZäöüÄÖÜß0-9]+)*', text_liste_Linke_4_str)
text_liste_Grüne_4_str_bereinigt = re.findall(r'[a-zA-ZäöüÄÖÜß0-9]+(?:-[a-zA-ZäöüÄÖÜß0-9]+)*', text_liste_Grüne_4_str)
text_liste_FDP_4_str_bereinigt = re.findall(r'[a-zA-ZäöüÄÖÜß0-9]+(?:-[a-zA-ZäöüÄÖÜß0-9]+)*', text_liste_FDP_4_str)

In [14]:
# Berechnung der Anzahl der Wörter pro Partei im 4. Zeitraum

text_liste_CDU_4_str_bereinigt = len(text_liste_CDU_4_str_bereinigt)
text_liste_SPD_4_str_bereinigt = len(text_liste_SPD_4_str_bereinigt)
text_liste_AfD_4_str_bereinigt = len(text_liste_AfD_4_str_bereinigt)
text_liste_Linke_4_str_bereinigt = len(text_liste_Linke_4_str_bereinigt)
text_liste_Grüne_4_str_bereinigt = len(text_liste_Grüne_4_str_bereinigt)
text_liste_FDP_4_str_bereinigt = len(text_liste_FDP_4_str_bereinigt)

In [15]:
#  Erstellung DataFrame mit der Anzahl aller Reden pro Zeitraum pro Partei
# sowie der Anzahl der Wörter für alle Reden pro Partei im 4. Zeitraum + Anzahl der Wörter im Wahlprogramm

cdu = [text_liste_CDU_1_no,text_liste_CDU_2_no,text_liste_CDU_3_no,text_liste_CDU_4_no,text_liste_CDU_4_str_bereinigt,CDU_WP_anzahl,text_liste_CDU_5_no]
spd = [text_liste_SPD_1_no,text_liste_SPD_2_no,text_liste_SPD_3_no,text_liste_SPD_4_no,text_liste_SPD_4_str_bereinigt,SPD_WP_anzahl,text_liste_SPD_5_no]
afd = [text_liste_AfD_1_no,text_liste_AfD_2_no,text_liste_AfD_3_no,text_liste_AfD_4_no,text_liste_AfD_4_str_bereinigt,AfD_WP_anzahl,text_liste_AfD_5_no]
linke = [text_liste_Linke_1_no,text_liste_Linke_2_no,text_liste_Linke_3_no,text_liste_Linke_4_no,text_liste_Linke_4_str_bereinigt,Linke_WP_anzahl,text_liste_Linke_5_no]
grüne = [text_liste_Grüne_1_no,text_liste_Grüne_2_no,text_liste_Grüne_3_no,text_liste_Grüne_4_no,text_liste_Grüne_4_str_bereinigt,Grüne_WP_anzahl,text_liste_Grüne_5_no]
fdp = [text_liste_FDP_1_no,text_liste_FDP_2_no,text_liste_FDP_3_no,text_liste_FDP_4_no,text_liste_FDP_4_str_bereinigt,FDP_WP_anzahl,"0"]

df_anzahl = pd.DataFrame({"CDU/CSU":cdu,"SPD":spd,"AfD":afd,"Die Linke":linke,"Die Grünen":grüne,"FDP":fdp,})
df_anzahl = df_anzahl.rename(index={
    0: "Reden 01.01.-31.03.22",
    1: "Reden 01.10.-31.12.23",
    2: "Reden 01.10.-31.12.24",
    3: "Reden 01.01.-23.02.25",
    4: "    ...Anzahl Wörter",
    5: "(Wörter im Wahlprogramm)",
    6: "Reden 01.05.-31.07.25"
})
df_anzahl

,CDU/CSU,SPD,AfD,Die Linke,Die Grünen,FDP
Reden 01.01.-31.03.22,366,453,205,111,278,230
Reden 01.10.-31.12.23,587,677,312,199,481,320
Reden 01.10.-31.12.24,434,634,246,115,348,258
Reden 01.01.-23.02.25,88,114,47,24,92,58
...Anzahl Wörter,52728,63439,22776,10944,44045,34836
(Wörter im Wahlprogramm),25479,26252,26363,31719,47571,18688
Reden 01.05.-31.07.25,658,430,396,218,303,0


In [16]:
# in LaTex überführen

latex_code = df_anzahl.to_latex(index=False, caption='Reden', label='tab:meine_tabelle')
with open('tabelle_anzahl.tex', 'w') as f:
    f.write(latex_code)